# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library, referencing all data elements using their schema `@id`s.

### Dataset Source
FAIR² dataset, as described by its Croissant schema, is available at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Print title and description
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview

Review available record sets and fields using their `@id`s.

> **Note:** The Croissant schema uses `@id` fields to uniquely identify all entities. All references to record sets, fields, and columns will use their `@id`s.


In [ ]:
# Get the list of available record sets via their @id in the metadata
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in this dataset's Croissant schema!")
else:
    for rs in record_sets:
        print(f"Record set: {rs['@id']} -- {rs.get('name', '[no name]')}")
        if 'field' in rs:
            print(" Fields:")
            for field in rs['field']:
                # field could be a dict or a string (reference). Dereference if needed:
                if isinstance(field, dict):
                    print(f"    {field['@id']} -- {field.get('name', '[no name]')}")
                elif isinstance(field, str):
                    print(f"    {field}")
        print("")
# Otherwise: The FAIR2 schema as hosted by Sen does not populate record sets in the Croissant metadata's 'recordSet' property.
# We will attempt to enumerate records using mlcroissant's .records(), which should list all available record sets if not specified.
if not record_sets:
    print("Attempting to infer record sets from dataset.records().")
    print("Available record set @id's (use the first one for demonstration):")
    all_record_sets = dataset.record_set_ids
    for rsid in all_record_sets:
        print(f"- {rsid}")

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames for analysis. Use the record set and field `@id`s identified above.


In [ ]:
# As discovered above, pick available record set IDs to extract data
# Use dataset.record_set_ids to get all valid record sets
record_sets = dataset.record_set_ids
dataframes = {}

for record_set_id in record_sets:
    df = pd.DataFrame(dataset.records(record_set=record_set_id))
    if df.shape[0] > 0:
        dataframes[record_set_id] = df
        print(f"Loaded {df.shape[0]} rows from record set {record_set_id}")
    else:
        print(f"No data for record set {record_set_id}")

# Pick the first available record set with data
primary_rs_id = next(iter(dataframes.keys()))
print(f"\nColumns for record set {primary_rs_id}:")
print(dataframes[primary_rs_id].columns.tolist())
dataframes[primary_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply data processing operations using `@id`s for both record set and fields/columns, such as filtering, normalizing, and grouping. The actual field `@id`s will depend on the schema and loaded DataFrame. Replace the placeholders below with the IDs as discovered above.


In [ ]:
# Let's inspect the DataFrame to identify numeric fields by @id
df = dataframes[primary_rs_id]
print("Field (column) @id's and dtype:")
print(df.dtypes)

# Choose a numeric field for demonstration, such as an age, interval, or other measurement
# We'll simply use the first numeric column available
numeric_fields = df.select_dtypes(include=['number']).columns.tolist()
if numeric_fields:
    numeric_field = numeric_fields[0]
    print(f"Selected numeric field for EDA: {numeric_field}")

    threshold = df[numeric_field].mean()  # use mean for demonstration
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Identify a categorical or group field (e.g., sex, diagnosis, etc.) by @id
    # Try to pick the first field with dtype=object that is not all unique (likely a category)
    object_fields = [c for c in df.select_dtypes(include=['object']).columns if df[c].nunique() < len(df)/2]
    if object_fields:
        group_field = object_fields[0]
        print(f"\nGrouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No numeric fields found in the selected record set.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset using their `@id`s.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field (if exists)
if 'numeric_field' in locals() and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} in record set {primary_rs_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field exists, show boxplot/grouped distribution
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we've demonstrated how to load, explore, and process the FAIR² dataset package using the `mlcroissant` library, referencing all structural elements using their schema `@id`s for reproducibility. We inspected available record sets, loaded them into DataFrames, and performed basic EDA including filtering, normalization, and group-wise aggregation—followed by visualization of key numeric fields. This Croissant-powered workflow ensures your data exploration is fully consistent with the dataset schema and remains robust to future schema evolution.
